In [1]:
from ExponentialBet import ExponentialBet
from utils import prepare_exponential_parameters



etas = prepare_exponential_parameters(0.01, 1, 10)

exp_bet = ExponentialBet(etas)

In [3]:
import numpy as np
n = 5000
rho = 0.7
beta = 2.0
gamma = 1.0
sigma = 1.0

# Z
Z = np.random.randn(n)

# X | Z
X = rho * Z + np.sqrt(1 - rho**2) * np.random.randn(n)

# response
Y = beta * X + gamma * Z + sigma * np.random.randn(n)

K = 100

X_tilde = (
    rho * Z[:, None]
    + np.sqrt(1 - rho**2) * np.random.randn(n, K)
)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.5, random_state=0)

model = LinearRegression()
model.fit(X_train, y_train)

ValueError: Expected 2D array, got 1D array instead:
array=[ 0.69106893  0.32989053  0.50826469 ... -1.48075573 -0.23665392
  0.1820748 ].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [8]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# import your class
# from your_module import ExponentialBet


# ----------------------------
# 1. DATA GENERATION
# ----------------------------

def generate_data(n=5000, d=5, rho=0.6, beta_strength=2.0, noise=1.0, seed=0):
    np.random.seed(seed)

    Sigma = rho * np.ones((d, d)) + (1 - rho) * np.eye(d)

    X = np.random.multivariate_normal(np.zeros(d), Sigma, size=n)

    j = 0

    beta = np.zeros(d)
    beta[j] = beta_strength

    gamma = np.ones(d)
    gamma[j] = 0.0

    eps = noise * np.random.randn(n)

    Y = X @ beta + X @ gamma + eps

    return X, Y, Sigma, j


# ----------------------------
# 2. EXACT CONDITIONAL SAMPLER
# ----------------------------

def sample_X_tilde(X_minus_j, j, Sigma, K=50):
    idx = np.arange(Sigma.shape[0]) != j

    Sigma11 = Sigma[j, j]
    Sigma12 = Sigma[j, idx]
    Sigma22 = Sigma[np.ix_(idx, idx)]

    Sigma22_inv = np.linalg.inv(Sigma22)

    mu = X_minus_j @ (Sigma12 @ Sigma22_inv).T
    var = Sigma11 - Sigma12 @ Sigma22_inv @ Sigma12.T

    noise = np.random.randn(len(X_minus_j), K) * np.sqrt(var)

    return mu[:, None] + noise


# ----------------------------
# 3. SETUP DATA
# ----------------------------

X, Y, Sigma, j = generate_data(beta_strength=2.0)

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.5, random_state=0
)


# ----------------------------
# 4. TRAIN LINEAR MODEL
# ----------------------------

model = LinearRegression()
model.fit(X_train, y_train)




LinearRegression()

In [11]:
# ----------------------------
# 5. YOUR REAL STRATEGY
# ----------------------------

from ExponentialBet import ExponentialBet  # adjust import

strategy = ExponentialBet(
    parameters=np.array([0.5, 1.0, 2.0]),
    exact=True
)


# IMPORTANT: initialize state (if needed in your base class)
#strategy.past_martingales = np.ones(len(strategy.parameters))


# ----------------------------
# 6. SEQUENTIAL LOOP
# ----------------------------

batch = 5
K = 50

n = len(X_test)

wealth_path = []

for t in range(0, n - batch, batch):

    X_batch = X_test[t:t+batch]
    y_batch = y_test[t:t+batch]

    X_minus_j = np.delete(X_batch, j, axis=1)

    # conditional samples of X_j | X_-j
    X_tilde = sample_X_tilde(X_minus_j, j, Sigma, K=K)

    # YOUR REAL CALL
    w = strategy.wealth(
        model=model,
        x=X_batch[:, [j]],
        x_tildes=X_tilde,
        y=y_batch,
        z=X_minus_j
    )

    strategy.update()

    wealth_path.append(strategy.past_martingales.copy())


# ----------------------------
# 7. RESULTS
# ----------------------------

wealth_path = np.array(wealth_path)

print("Final wealth vector:", wealth_path[-1])
print("Mean wealth:", wealth_path.mean(axis=0))

AxisError: axis 1 is out of bounds for array of dimension 1

In [2]:
exp_bet.e_value()